# ViT adaptation demo (Flowers102 few-shot)

Quick sanity checks for the TPA-17 pipeline. Full training and the
comparison report are driven by `python -m src.vit.main sweep`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "vit" / "main.py").exists()
)
sys.path.insert(0, str(ROOT))

from src.vit.config import Config
from src.vit.dataset import build_datasets

cfg = Config(method="lp", shots_per_class=5, batch_size=8, num_workers=0)
processor, train_ds, train_eval_ds, val_ds, test_ds = build_datasets(cfg)
print("sizes:", len(train_ds), len(val_ds), len(test_ds))
print("first ten train labels:", train_ds.labels[:10])

In [ ]:
from src.vit.metrics import accuracy, top_k_accuracy
from src.vit.trainer import ViTClassifier

for method in ["full", "lp", "lora", "bitfit", "ncm"]:
    model = ViTClassifier(
        method=method,
        num_classes=cfg.num_classes,
        metric_fns={"acc": accuracy, "top5": top_k_accuracy},
    )
    counts = model.parameter_counts()
    print(
        f"{method:6s} trainable={counts['trainable_parameters']:>10,} "
        f"total={counts['total_parameters']:>10,} "
        f"({100 * counts['trainable_fraction']:.2f}%)"
    )
    del model

In [ ]:
batch = next(iter(__import__("torch").utils.data.DataLoader(train_ds, batch_size=4)))
print({key: tuple(value.shape) for key, value in batch.items()})